# EnvGuard AI: Anomaly Detection Preprocessing Pipeline
## Zero-Knowledge Secrets Manager Behavioral Telemetry Analysis
- **Dataset:** CERT Insider Threat Dataset r4.2 (Zenodo DOI: `10.5281/zenodo.19435851`)  
- **Target Model:** Unsupervised Isolation Forest  

---
### Objective
This notebook demonstrates the complete data inspection, cleaning, feature selection, behavioral feature engineering, and preparation of training and evaluation feature matrices for Isolation Forest anomaly detection in EnvGuard.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to python path
sys.path.append(os.path.abspath(".."))
from src.preprocessing import clean_data, engineer_features, select_features, split_data_chronologically

pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 1000)
%matplotlib inline


## 1. Dataset Loading & Exploratory Inspection

In [ ]:
raw_dataset_path = os.path.join("..", "data", "raw", "feature_matrix_v1.parquet")
df = pd.read_parquet(raw_dataset_path, engine="fastparquet")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
df.head(3)


In [ ]:
# Check missing values and duplicates
missing_counts = df.isnull().sum()
missing_df = pd.DataFrame({"Missing Count": missing_counts, "Percentage (%)": (missing_counts / len(df)) * 100})
print("Columns with Missing Values:")
print(missing_df[missing_df["Missing Count"] > 0])

print(f"\nExact Full Duplicates: {df.duplicated().sum()}")
print(f"User-Day Duplicates: {df.duplicated(subset=['user', 'day']).sum()}")


## 2. Feature Selection & Zero-Knowledge Rationale
EnvGuard monitors access using audit-log metadata only. We explicitly exclude:
1. **Psychometrics (OCEAN):** `O`, `C`, `E`, `A`, `N` (Personality scores do not exist in audit logs).
2. **Email Sentiment:** `avg_sentiment`, `sentiment_negative_flag`, `sentiment_strong_negative_flag`, `sentiment_30d_std`, `sentiment_z_user` (Zero-knowledge secret managers do not inspect email text).
3. **Static Demographics:** `role`, `department`, `functional_unit` (~10% missing, static non-behavioral).
4. **Direct Identifiers:** `user`, `day` are preserved in metadata tables, but excluded from the model feature matrix to prevent memorization.


## 3. Data Cleaning & Feature Engineering

In [ ]:
# 1. Clean boundary missing values in logon_7d_mean
df_clean = clean_data(df)

# 2. Engineer domain-specific behavioral features
df_eng = engineer_features(df_clean)

# 3. Select numerical features and isolate metadata
metadata_df, features_df = select_features(df_eng)
print(f"Processed Feature Matrix Shape: {features_df.shape}")
features_df.info()


## 4. Chronological Train / Evaluation Split
To prevent temporal data leakage, we partition the dataset chronologically:
- **Training Baseline (80%):** 2010-01-02 to 2011-02-05 (268,979 samples)
- **Evaluation Holdout (20%):** 2011-02-06 to 2011-05-17 (61,473 samples)


In [ ]:
splits = split_data_chronologically(metadata_df, features_df, train_ratio=0.8)
print(f"Cutoff Date: {splits['split_date']}")
print(f"X_train Shape: {splits['X_train'].shape}")
print(f"X_eval Shape: {splits['X_eval'].shape}")


## 5. Visualizations & Distributions

In [ ]:
# Distribution of engineered sensitive resource activity
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(features_df["total_resource_activity"], bins=30, ax=ax, color="#d95f02", log_scale=(False, True))
ax.set_title("Engineered Total Sensitive Resource Activity (File + USB Operations) [Log Scale]", fontweight="bold")
ax.set_xlabel("Resource Activity Count")
ax.set_ylabel("Log Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# User Baseline Deviation Z-Scores (Surge Detection)
fig, ax = plt.subplots(figsize=(10, 4))
z_cols = ["total_logon_activity_z_user", "file_activity_count_z_user", "usb_activity_count_z_user", "max_user_activity_z"]
sns.boxplot(data=features_df[z_cols], ax=ax, palette="Set2")
ax.axhline(3, color="orange", linestyle=":", label="Moderate Surge (Z > 3)")
ax.axhline(6, color="red", linestyle="--", label="Extreme Surge (Z > 6)")
ax.set_title("User Deviation Z-Scores Across Activity Channels", fontweight="bold")
ax.set_ylabel("Standard Deviations (Z)")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Output Verification & Summary
The preprocessed outputs have been persisted to:
- `../data/processed/processed_features.parquet` (Complete feature table with user/day metadata)
- `../data/processed/training_features.parquet` (Numerical training matrix for Isolation Forest)
- `../data/processed/evaluation_features.parquet` (Numerical evaluation matrix for test inference)

Notice: Ground-truth target labels are not included in the Zenodo CERT feature matrix v1; therefore, no synthetic or empty `labels.parquet` was generated.
